In [ ]:
import re
from pathlib import Path
import pandas as pd
from data_processing.arc_paths import get_report_root

In [ ]:
def interval_from_string(interval_str: str) -> pd.Interval:
    pattern = re.compile(r"([\(\[])(.*),\ *(.*)([\)\]])")
    match_result = pattern.match(interval_str)
    if match_result is None:
        raise ValueError("Interval string could not be parsed")
    left_bracket, left_str, right_str, right_bracket = match_result.groups()

    left_closed = left_bracket == '['
    right_closed = right_bracket == ']'
    left = float(left_str)
    right = float(right_str)

    closed = 'neither'
    if left_closed and right_closed:
        closed = 'both'
    elif left_closed:
        closed = 'left'
    elif right_closed:
        closed = 'right'
    
    return pd.Interval(left, right, closed=closed)

In [ ]:
exp_id = "ID-484"
file_name = f"{exp_id}_gamma_spectrum_60s_time_bin.csv"

exp_root = get_report_root(exp_id)
gamma_spec_path = exp_root / file_name

In [ ]:
df = pd.read_csv(gamma_spec_path)

In [ ]:
df = df.drop(["Time Bin"], axis=1).transpose()
df

In [ ]:
new_df = pd.DataFrame(df.sum(axis=1))
new_df = new_df.rename(columns={0: "Count"}).reset_index(names="Light output interval (MeVee)")
new_df

In [ ]:
l_col = new_df["Light output interval (MeVee)"]
l_col = l_col.apply(interval_from_string)
new_df["Light output interval (MeVee)"] = l_col
new_df

In [ ]:
l_series = new_df['Light output interval (MeVee)']
new_df['Light output bin start (MeVee)'] = l_series.map(lambda x: x.left)
new_df['Light output bin end (MeVee)'] = l_series.map(lambda x: x.right)
new_df['Light output midpoint (MeVee)'] = l_series.map(lambda x: x.mid)
new_df

In [ ]:
new_df = new_df.drop(columns=['Light output interval (MeVee)'])
new_df

In [ ]:
export_path = Path() / f"{exp_id}_simple_gamma_spectrum.csv"
new_df.to_csv(export_path, index=False, columns=[
    "Light output bin start (MeVee)",
    "Light output bin end (MeVee)",
    "Light output midpoint (MeVee)",
    "Count"
])